In [1]:
!pip install -qU sentence-transformers

In [2]:
!pip install psycopg2-binary

In [3]:
import random
import pandas as pd
from sentence_transformers import SentenceTransformer
import os
import psycopg2

os.environ['WANDB_DISABLED'] = 'true'

In [4]:
import zipfile
from pathlib import Path

# загрузка ретривера
zip_path = Path("e5_custom (2).zip")
extract_to = Path("e5_custom")

extract_to.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_to)

print("Готово:", extract_to.resolve())

In [5]:
# 3. Инициализация модели
model = SentenceTransformer("e5_custom/kaggle/working/e5_custom")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
def to_pgvector(vec):
    return "[" + ",".join(str(float(x)) for x in vec) + "]"

In [7]:
# функция поиска релевантных чанков

def semantic_search(query, conn, model, top_k=12):
    try:
        query_embedding = model.encode_query([query], normalize_embeddings=True)[0]
    except AttributeError:
        prefixed_query = f"query: {query}"
        query_embedding = model.encode([prefixed_query], normalize_embeddings=True)[0]

    query_embedding_sql = to_pgvector(query_embedding)

    cur = conn.cursor()
    cur.execute(
        """
        SELECT id, source_doc, chunk, vector <=> %s::vector AS distance
        FROM chunks
        WHERE vector IS NOT NULL
        ORDER BY vector <=> %s::vector
        LIMIT %s;
        """,
        (query_embedding_sql, query_embedding_sql, top_k * 3)
    )
    rows = cur.fetchall()
    cur.close()

    # Убираем дубликаты
    seen = set()
    unique_results = []

    for row_id, source_doc, chunk, distance in rows:
        if chunk not in seen:
            seen.add(chunk)
            unique_results.append({
                "id": row_id,
                "source_doc": source_doc,
                "chunk": chunk,
                "distance": float(distance),
            })
        if len(unique_results) >= top_k:
            break

    return unique_results

In [8]:
# conn.rollback()
# print("rollback done")

In [10]:
conn = psycopg2.connect(
    host="127.0.0.1",
    port=5433,
    database="ragdatabase",
    user="raguser",
    password="ragpassword"
)

In [11]:
results = semantic_search(
    query="Чем отличается установка IM101 в крейте расширения для резервированного и одиночного процессора?",
    conn=conn,
    model=model,
    top_k=3
)
for r in results:
    print(r["distance"], r["chunk"])

0.15643564049560155 2.8 Интеграция Контроллера « El100» с подсистемой ввода -вывода Контроллера «El -200»

Контроллер «El -100» , как в одиночном, так и в дублированном исполнении процессорного модуля, имеет  средства,  обеспечивающие  его  подключение  и совместную  работу  с  крейтами расширения модулей УСО контроллера Elicont -200 (АДИГ.421457.012).

При  необходимости  использования  подсистемы  ввода -вывода  Контроллера  «El -200»  в сочетании с дублированным исполнением процессора Контроллера «El -100» интеграция производится  путём  сопряжения  крейтов  CA 20 X расширения  УСО  Контроллера  «El -200»  с крейтом дублированного Процессора  CA 12 X Контроллера «El -100» (при использовании дублированных  процессорных  модулей  шина  INEL также  дублируется).  В  крейт  расширения CA 20 X устанавливаются два модуля IM 201 (либо IM202), а подключение интерфейсных кабелей INEL осуществляется к модулям CM101 крейта  CA 12 X Контроллера «El -100». Схемы подключения интерфейсных кабелей 

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

In [13]:
from openai import OpenAI

client7 = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key 
  )

def generate_answer(question):
    # Ищем релевантные абзацы
    search_results = semantic_search(question, top_k=12, conn=conn, model=model)
    if not search_results:
        return "Не удалось найти подходящую информацию"

    # Промпт
    prompt = f"""
    Задача:
    Найди в предоставленных данных ответ на вопрос пользователя. Затем сформулируй ответ на вопрос пользователя.
    Ответ должен звучать как экспертное утверждение, без упоминания источников, данных или контекста. Предоставь максимум информации используя имеющиеся данные.
    Информации может быть мало, в таком случае скажи "Недостаточно информации".

    Вопрос пользователя: {question}
    Данные для ответа:
    1. {search_results[0]["chunk"]}
    2. {search_results[1]["chunk"]}
    3. {search_results[2]["chunk"]}

    """

    # Запрос к LLM
    completion = client7.chat.completions.create(
      extra_body={},
      model="openrouter/elephant-alpha",
      messages=[
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": prompt
            }
          ]
        }
      ]
    )

    # Обработка ответа
    if completion and completion.choices:
        content = completion.choices[0].message.content
        return f"""
        Ответ: {content}\n\n
        Источники: \n\n{search_results[0]["chunk"]}\n\n{search_results[0]["chunk"]}\n\n{search_results[2]["chunk"]}
        """
    else:
        print("Ошибка: LLM не вернул ответ")

In [14]:
generate_answer("Чем отличается установка IM101 в крейте расширения для резервированного и одиночного процессора?")

'\n        Ответ: Установка IM101 в крейте расширения для резервированного процессора требует размещения двух модулей IM101 на крейтах CA 20 X конкретного исполнения, обеспечивая дублирование шины INEL, в то время как для одиночного процессора устанавливается только один модуль IM101 на втором посадочном месте крейта CA 12 X, с неиспользуемым первым местом; это различие определяется структурой шины INEL, схемами подключения и требованиями к конфигурированию крейтов в зависимости от режима работы процессора.\n\n\n        Источники: 2.3 Компоновка крейта расширения УСО\n\nПроцессор с помощью модулей УСО принимает по физическим линиям связи сигналы от датчиков  объекта  управления  и  выдает  команды  на  его  исполнительные  устройства.  Типы датчиков и нагрузок, с которыми работает Контроллер «El -100 »,  типы и диапазоны их сигналов приведены в таблице 15 настоящего РЭ.\n\nВсе модули УСО являются интеллектуальными аппаратно -программными устройствами.\n\nПроектная  компоновка  модулей 